### Messages

Messages are the fundamenta unit of context for models in langchain. they represent the input and output of models, carrying boyh the content and metadata needed to represent the state of a converstation when interecting with an llm messages are object that contain:

- Role - Identifyies the messages type (e.g.  system, user)
- Content - Represents the actual content of messages (like text, images, audio, documents, etc.)
- Metadata - Optinal field such as response information, message IDs, and token usage

Lanchain provide a standard messag3es type that works across all model provides, ensuring consistent behaviour regardless of the model being called.



In [6]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

In [7]:
model.invoke("please tell me something about AI ?")

AIMessage(content='<think>\nOkay, the user asked for something about AI. Let me start by understanding what they might be looking for. Since they just said "please tell me something about AI," they could be a beginner wanting a general overview or someone with more specific interests. I should cover the basics first.\n\nFirst, define AI. Artificial Intelligence, right? A field of computer science focused on creating systems that can perform tasks requiring human-like intelligence. Then maybe mention the main areas like machine learning, natural language processing, computer vision. But I need to explain these in simple terms without getting too technical.\n\nNext, history. When did AI start? The Dartmouth Conference in 1956 is a key point. Then the different waves of AI development, like the early rule-based systems, the AI winter, resurgence with machine learning and big data. Maybe touch on recent advances like deep learning.\n\nApplications are important too. Where is AI used today?

### Text Prompts

Text Promtps are strings - ideal for straightforward generation tasks where you don't need to ratain conversation history.

Use text prompts when : 
- You have a single, standalone request
- You don't need cconversation history
- You want minimal code compelxity 

### Message Prompts

Alternatively you can pass in a list of messages to the model by providing a list of message objects.

Message types
- System messages - Tells the model how to behave and provide context for interactions
- Human messages - Represents user input and interactions with the model.
- AI message - Responses generated by the model, including text content, tool calls, and metadata
- tool messages- represents the output of tool calls

In [8]:
 #details info to llm with system Message

from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
you are a senior python developer with expertise in web frameworks. 
Always provide code examples and explain your reasoning.
Be concise but throught in your explainations.
""")

messages = [system_msg,
HumanMessage("How do I create a REST API?")]

response = model.invoke(messages)
print(response.content)

<think>
Okay, the user is asking how to create a REST API. Let me start by recalling the basics. REST APIs are typically built using frameworks like Flask or Django in Python. Since the user might be looking for a straightforward example, Flask is probably a better choice here because it's lightweight and easy to get started with.

First, I should outline the steps. They'll need to install Flask, set up a basic server, define routes, handle different HTTP methods, and maybe add some request parsing. Let me think about the example. A common example is a simple resource like a book. So, creating endpoints to get all books, get a single book by ID, add a new book, update a book, and delete a book.

Wait, how do I handle JSON data in Flask? Oh right, using request.get_json() to parse incoming JSON. Also, for sending JSON responses, just return a dictionary and Flask will convert it. Also, need to import necessary modules like Flask, request, and maybe jsonify. Wait, actually, in newer vers

In [9]:
## Message Metadata

human_msg = HumanMessage(
    content="Hello",
    name = "alice",
    id = "msg_123",
)

In [10]:
response = model.invoke([human_msg])
response

AIMessage(content='<think>\nOkay, the user said "Hello". I need to respond appropriately. Let me think about the right way to reply.\n\nFirst, a simple greeting. Maybe "Hi there!" or "Hello!". Then, offer assistance. The user might need help with something specific. I should ask how I can assist them today. Keep it friendly and open-ended. Let them know I\'m here for any questions or tasks. Make sure the response is concise but welcoming. Avoid using any technical jargon. Just a straightforward, approachable message. Alright, that should cover it.\n</think>\n\nHi there! How can I assist you today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 130, 'prompt_tokens': 9, 'total_tokens': 139, 'completion_time': 0.2530579, 'completion_tokens_details': None, 'prompt_time': 0.000240307, 'prompt_tokens_details': None, 'queue_time': 0.05322113, 'total_time': 0.253298207}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'o

In [11]:
from langchain.messages import  AIMessage, SystemMessage, HumanMessage

ai_msg = AIMessage("i'd be happy yo help you with that question")
messages = [
    SystemMessage("your are a helpful assistant"),
    HumanMessage("can you help me ?"),
    ai_msg,
    HumanMessage("Great what is 2 + 2")
]
response = model.invoke(messages)
print(response.content)


<think>
Okay, the user asked "what is 2 + 2" after I said I can help. Let me see... 2 plus 2 is a basic math problem. The answer is 4, right? But wait, maybe they want more than just the answer? Maybe they need an explanation? Let me check the history.

Looking at the history, the user said "can you help me?" and I replied "i'd be happy yo help you with that question". Then they asked "Great what is 2 + 2". The user might be testing if I can do basic math, or maybe they're just starting to learn. Since the question is straightforward, the answer is 4. But maybe I should confirm if they need a step-by-step explanation or if they want to know different ways to solve it. But the user hasn't asked for that yet. Maybe just give the answer and ask if they need more help. Let me make sure I'm not overcomplicating. The answer is 4. Keep it simple.
</think>

The answer to 2 + 2 is **4**. 

If you need further clarification or have more questions, feel free to ask! 😊


In [12]:
response.usage_metadata

{'input_tokens': 51, 'output_tokens': 249, 'total_tokens': 300}

In [13]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name":"get_weather",
        "args": {"location":"san Francisco"},
        "id": "call_123"
    }]
)

In [15]:
weather_result = "Sunny, 72f"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"
)

In [16]:
messages = [
    HumanMessage("what is the weather in san francisco"),
    ai_message,
    tool_message,
]
response = model.invoke(messages)


In [19]:
response

AIMessage(content="<think>\nOkay, the user asked for the weather in San Francisco, and the response came back as sunny with 72°F. I need to present this information clearly. Let me check if there's any additional details I should include, like wind speed or humidity, but the tool only provided the basic info. Maybe mention that it's a good day for outdoor activities since it's sunny. Also, confirm the temperature in both Fahrenheit and Celsius for clarity. Wait, the user might not need Celsius, but it's good to offer. Let me structure the response to be friendly and informative. Make sure to keep it concise but helpful.\n</think>\n\nThe current weather in San Francisco is **sunny** with a temperature of **72°F (22°C)**. It looks like a great day for outdoor activities! Let me know if you'd like more details. ☀️", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 179, 'prompt_tokens': 57, 'total_tokens': 236, 'completion_time': 0.359931009, 'completion_tokens